# TP1 — Le réseau du métro de Montréal
## Carnet, graphes, Dijkstra — et un premier modèle

**Nom :** _Louis-Philippe Leboeuf_ &nbsp;&nbsp;&nbsp; **Coéquipier :** _--_ &nbsp;&nbsp;&nbsp; **Groupe :** _07979_

---

Ce TP se fait **seul ou en équipe de 2**. Une équipe remet **un seul carnet** et
indique à la fin qui a fait quoi.

**Chaque question, y compris les questions de code, se termine par une cellule
d'observation en Markdown.** Écrivez-y ce que vous constatez, en phrases complètes,
en français. Une cellule d'observation laissée telle quelle vaut 0 pour la question.

Avant de remettre : **Kernel → Redémarrer et tout exécuter**.

## Partie 2 — Charger et explorer le réseau

Le fichier `donnees/metro_montreal.csv` contient une ligne par lien entre deux
stations voisines, avec la durée approximative en minutes. Le fichier
`donnees/stations.csv` donne la ligne de chaque station.

In [1]:
import csv
#  Veuillez choisir une station de départ et d'arrivée dans les choix suivant: Acadie, Angrignon, Assomption, Atwater, Beaubien, Beaudry, Berri-UQAM, Bonaventure, Cadillac, Cartier, Champ-de-Mars, Charlevoix, Cote-Sainte-Catherine, Cote-Vertu, Cote-des-Neiges, Cremazie, D'Iberville, De Castelnau, De l'Eglise, De la Concorde, De la Savane, Du College, Edouard-Montpetit, Fabre, Frontenac, Georges-Vanier, Guy-Concordia, Henri-Bourassa, Honore-Beaugrand, Jarry, Jean-Drapeau, Jean-Talon, Jolicoeur, Joliette, LaSalle, Langelier, Laurier, Lionel-Groulx, Longueuil, Lucien-L'Allier, McGill, Monk, Mont-Royal, Montmorency, Namur, Outremont, Papineau, Parc, Peel, Pie-IX, Place-Saint-Henri, Place-d'Armes, Place-des-Arts, Plamondon, Prefontaine, Radisson, Rosemont, Saint-Laurent, Saint-Michel, Sauve, Sherbrooke, Snowdon, Square-Victoria, Universite-de-Montreal, Vendome, Verdun, Viau, Villa-Maria

reseau = {}
depart = "Laurier"
arrivee = "Namur"

with open("donnees/metro_montreal.csv", encoding="utf-8") as fichier:
    for ligne in csv.DictReader(fichier):
        a = ligne["station_a"]
        b = ligne["station_b"]
        minutes = int(ligne["minutes"])
        reseau.setdefault(a, []).append((b, minutes))
        reseau.setdefault(b, []).append((a, minutes))

infos_station = {}
with open("donnees/stations.csv", encoding="utf-8") as fichier:
    for ligne in csv.DictReader(fichier):
        infos_station[ligne["station"]] = ligne




def voisins(reseau, station):
    return [nom for nom, minutes in reseau[station]]




def temps(reseau, a, b):
    for nom, minutes in reseau[a]:
        if nom == b:
            return minutes
    return None







**Q2.1** — Affichez le nombre de stations et le nombre de **liens**.

In [2]:
# Q2.1
print(len(reseau), "stations")
print(round(sum(len(voisins(reseau, station)) for station in reseau)/2),"Liens")


68 stations
69 Liens


> **Observation — obligatoire.** Comparez vos deux nombres. Qu'est-ce que leur écart vous dit sur la forme du réseau ?
>
>

**Q2.2** — Trouvez la ou les stations qui ont le plus de voisins, puis la liste des
**terminus**.

In [3]:
# Q2.2
Boutdtrajet =[]
multivoisins= 0
voisin = []
for station in reseau:
    if len(voisins(reseau, station)):
        #print(station,len(voisins(reseau, station)))
        voisin.append(station)
    if len(voisins(reseau, station)) ==1:
        Boutdtrajet.append(station)
    if len(voisins(reseau, station)) >= multivoisins:
        #le deuxieme if remet la liste vide si il y a une nouvelle station avec plus de voisins.  Cela permet de garder toute les  stations avec le plus de voisins seulement dans la liste
        if len(voisins(reseau, station))>multivoisins:
            listevoisins=[]
        superstation = station
        multivoisins = len(voisins(reseau, station))
        listevoisins.append([superstation, multivoisins])

print("Les Termminus sont",Boutdtrajet)
print("La/Les station(s) qui a/ont le plus de voisins",listevoisins)




Les Termminus sont ['Angrignon', 'Honore-Beaugrand', 'Cote-Vertu', 'Montmorency', 'Saint-Michel', 'Longueuil']
La/Les station(s) qui a/ont le plus de voisins [['Berri-UQAM', 5]]


> **Observation — obligatoire.** Pourquoi ces stations-là ? Expliquez ce que leur nombre de voisins traduit dans le vrai réseau.
>
> *Les stations retenues pour les terminus sont celle qui n'ont qu'un seul voisin*
> *Un test a été fait pour voir si plusieurs stations avaient le meme nombre de voisins pour le plus grand nombre de voisins et ils sont tous ajouté à la liste*

**Q2.3** — Comptez combien de tronçons durent 1, 2 et 3 minutes.

In [4]:
# Q2.
liste1=[]
liste2=[]
liste3=[]
lesvoisins=[]
for station in reseau:
    for voisin in voisins(reseau, station):
        lesvoisins.append([station, voisin])
        if [voisin,station] in lesvoisins:
            continue
        if temps(reseau, station, voisin)==1:
            liste1.append([station, voisin])
        if temps(reseau, station, voisin)==2:
            liste2.append([station, voisin])
        if temps(reseau, station, voisin)==3:
            liste3.append([station, voisin])
print(len(liste1),"stations à 1 minute")
print(len(liste2),"stations à 2 minute")
print(len(liste3),"stations à 3 minute")


13 stations à 1 minute
37 stations à 2 minute
19 stations à 3 minute


> **Observation — obligatoire.** Quelle durée domine, et dans quelle proportion ? Gardez ce chiffre : il servira à la partie 5.
>
> *Les chemins à 2 minutes dominent par un peu plus de 50%*

## Partie 3 — Le parcours en largeur, et son arbre

**Q3.1** — Complétez `bfs(reseau, depart)`. Il rend `distance` (en nombre d'arrêts)
et `precedent`.

In [5]:
from collections import deque

# ce bloc contient plusieurs print qui peuvent être activé pour suivre ce qui se passe dedans le bfs.
# Le print qui est actif montre le resultat des ditances avec chaques stations de manière plus esthétique que de print le retour de
# distance, precedent.

def bfs(reseau, depart):
    distance = {depart: 0}
    precedent = {}
    file = deque([depart])
    visite = 0
    while file:
        #print("file",file)
        station = file.popleft()
        visite += 1
        #print("station#",visite," : La station",station,"est à",distance[station],"stations de la station",depart)
        #print("voisins:",voisins(reseau,station))
        for suivante in voisins(reseau, station):
           # print("suivante:",suivante)
           # print("voisins:",voisins(reseau,station))
           # print("precedent entre forif",precedent)
            if suivante not in distance:
                precedent[suivante] = station
                distance[suivante] = distance[station] + 1
                file.append(suivante)
              #  print("precedent",precedent)
              #  print("station distance",station,distance)
      #  print("je visite",station,"file:",file)


    return distance, precedent


> **Observation — obligatoire.** Décrivez en vos mots ce que contient `precedent` à la fin, et à quoi il servira.
>
> *Il contient chacun des liens qu'il a visité pour pouvoir refaire le chemin inverse apres*

**Q3.2** — Lancez le BFS depuis la station de votre choix. Combien de stations sont
atteintes ? Laquelle est la plus loin, en nombre d'arrêts ?

In [6]:
# Q3.2
# Cette manière de l'écrire sert à produire le graphique de Q3.3
distance, precedent = bfs(reseau,depart)


> **Observation — obligatoire.** Le nombre de stations atteintes vous apprend quelque chose sur le réseau : quoi ? Et la station la plus loin vous surprend-elle ?
>
> *Nous avons atteint les 68 stations, comptant même la station de départ qui est à distance de.. 0 de elle même.*

**Q3.3** — Dessinez l'**arbre de parcours** (les 3 premiers niveaux) avec
`graphviz`.

In [7]:
from graphviz import Digraph
# Q3.3
graphe = Digraph("Stations_métro")
for cle,valeur in precedent.items():
    graphe.edge(valeur,cle)


graphe.render("Stations_metro", format="png", view=True)


'Stations_metro.png'

> **Observation — obligatoire.** Le réseau n'est pas un arbre, et pourtant votre dessin en est un. Expliquez pourquoi.
>
> *Peu importe de où nous partons, avec le BFS, nous pouvons visiter toute les stations de manière linéaire et sans revenir en arrière.  Cela crée un arbre car si un chemin fait face à une station déjà visité, il s'arretera car il sait qu'il n'a pas à y retourner.*

## Partie 4 — Dijkstra, en minutes

**Q4.1** — Complétez `plus_petite_case`, `relacher` et `dijkstra`.

In [8]:
INFINI = 9999

def plus_petite_case(distance, barres):
    record = INFINI
    choisie = None
    for station in distance:
        if station not in barres and distance[station] < record:
            record = distance[station]
            choisie = station
    return choisie


def relacher(distance, precedent, a, b, minutes):
    nouvelle = distance[a] + minutes
    if nouvelle < distance[b]:
        distance[b] = nouvelle
        precedent[b] = a


def dijkstra(reseau, depart):
    distance = {station: INFINI for station in reseau}
    distance[depart] = 0
    precedent = {}
    barres = set()
    while len(barres) < len(reseau):
        a = plus_petite_case(distance, barres)
        barres.add(a)
        for b in voisins(reseau, a):
            relacher(distance, precedent, a, b,temps(reseau,a,b))
    return distance, precedent
distance,precedent = dijkstra(reseau,depart)

> **Observation — obligatoire.** À quoi sert l'ensemble `barres` ? Que se passerait-il si on l'oubliait ?
>
> *Il sert à ne pas repasser par une station que nous avons déjà passé par car si nous y revenons les chemins seront tous plus long à partir de ce point qu'avant.#*

**Q4.2** — Complétez `chemin` (la remontée des précédents) et `duree`.

In [9]:
def chemin(precedent, depart, arrivee):
    route = [arrivee]
    a = arrivee
    while depart != a:
        route.append(precedent[a])
        a = precedent[a]
    route.reverse()
    return route


def duree(reseau, route):
    total = 0
    for station in route:
        i = route.index(station)

        try:
            total += temps(reseau, route[i],route[i+1])
        except IndexError:
            return total


> **Observation — obligatoire.** `chemin` remonte à l'envers puis retourne la liste. Pourquoi ne peut-on pas la construire à l'endroit directement ?
>
> *(remplacez ce texte par vos phrases)*

**Q4.3** — Affichez le trajet le plus rapide entre deux stations de votre choix,
avec sa durée.

In [10]:
# Q4.3
route = chemin(precedent,depart,arrivee)
print("Le chemin partant de la station",route[0],"vers la station",route[len(route)-1])
print("Sera d'une durée de :",duree(reseau,route),"minutes")
print("et passera par les stations :")
print(route)

Le chemin partant de la station Laurier vers la station Namur
Sera d'une durée de : 29 minutes
et passera par les stations :
['Laurier', 'Mont-Royal', 'Sherbrooke', 'Berri-UQAM', 'Saint-Laurent', 'Place-des-Arts', 'McGill', 'Peel', 'Guy-Concordia', 'Atwater', 'Lionel-Groulx', 'Place-Saint-Henri', 'Vendome', 'Villa-Maria', 'Snowdon', 'Cote-Sainte-Catherine', 'Plamondon', 'Namur']


> **Observation — obligatoire.** Ce trajet correspond-il à celui que vous prendriez dans la vraie vie ? Si non, qu'est-ce qui manque à notre modèle ?
>
> *Surement que ce serait le chemin emprunté car c'est celui le moins long.  Il reste le facteur décisionnel humain qui peut, par exemple, le temps de transfert entre 2 lignes ou encore accepter de faire un trajet plus long en faisant moins de transferts et profitant du voyage pour faire un temps de lecture.*

## Partie 5 — Comparer les deux, et surligner le chemin

**Q5.1** — Écrivez `comparer(depart, arrivee)` qui affiche, pour chaque algorithme,
le nombre d'arrêts **et** la durée. Testez-la sur **trois** paires de stations, dont
**au moins une** où les deux algorithmes ne donnent pas le même trajet.

In [11]:
# Q5.1

def comparer(depart,arivee):
    distance_bfs, precedent_bfs = bfs(reseau,depart)
    BFSCHEMIN = chemin(precedent_bfs,depart,arivee)
    BFSDUREE = duree(reseau,BFSCHEMIN)
    print("BFS :",len(BFSCHEMIN),"arrêts pour une durée de:",BFSDUREE,"minutes")
    distance_dijkstra, precedent_dijkstra = dijkstra(reseau,depart)
    DIJKSTRASCHEMIN = chemin(precedent_dijkstra,depart,arivee)
    DIJKSTRADUREE = duree(reseau,DIJKSTRASCHEMIN)
    print("Dijkstra :",len(DIJKSTRASCHEMIN),"arrêts pour une durée de:",DIJKSTRADUREE,"minutes")
    if DIJKSTRADUREE < BFSDUREE:
        precedent = precedent_dijkstra
        distance = distance_dijkstra
        print("Dijkstra dit : un chemin plus rapide avec plus d'arrêts existe")
        return distance, precedent, precedent_bfs
    else:
        print("Le chemin le plus rapide est celui qui passe par le moins d'arrêts!! BONUS!!")
        return distance_bfs, precedent_bfs, precedent_bfs

> **Observation — obligatoire.** Sur vos trois paires, à quelle fréquence les deux sont-ils d'accord ? Reliez votre réponse au décompte des durées de la question 2.3.

> **Station Namur à Laurier :**
> *Le BFS donne 15 arrêts pour 33 minutes
> Dijkstra donne 18 arrêts pour 29 minutes*

> **Station Jolicoeur à Montmorency :**
> *Le BFS donne 25 arrêts pour 58 minutes
> Dijkstra donne 26 arrêts pour 48 minutes*

> **Station Longueuil à Outremont :**
> *Le BFS donne 13 arrêts pour 26 minutes
> Dijkstra donne 13 arrêts pour 26 minutes*

>**Resultat :**
*sur trois tests, un seul m'a donné le même résultat pour les deux*

**Q5.2** — Sur votre paire divergente, **surlignez le chemin de Dijkstra** dans le
dessin : les arêtes du chemin en **rouge et en épais**, la durée de chaque tronçon en
étiquette, et les stations du chemin remplies d'une couleur.

> **Les attributs à utiliser.** Sur une arête : `color="red"`, `penwidth="3"`,
> `fontcolor="red"`. Sur un sommet : `style="filled"`, `fillcolor="#ffd9d9"`.
> Le reste — quelles arêtes dessiner, comment reconnaître celles du chemin — est à
> vous.

In [12]:
# Q5.2
from graphviz import Digraph
# Q3.3
distance,precedent,precedent_bfs =comparer(depart,arrivee)
route = chemin(precedent,depart,arrivee)
routebfs = chemin(precedent_bfs,depart,arrivee)
graphe = Digraph("Meilleur Chemin")
for cle,valeur in precedent.items():
    labeltemps= str(temps(reseau,cle,valeur))
    if  cle in route:
        graphe.edge(valeur,cle,label=f"   {labeltemps} minutes",color="red", penwidth="3",fontcolor="red")
        graphe.node(depart, style="filled", fillcolor="#ffd9d9")
        graphe.node(cle, style="filled", fillcolor="#ffd9d9")
    elif cle not in route:
        graphe.edge(valeur,cle)
for key,value in precedent_bfs.items():
    if  key in routebfs:
        graphe.edge(value,key,label="BFS",color="blue", penwidth="3",fontcolor="blue")



graphe.render("Meilleur Chemin", format="png", view=True)


BFS : 15 arrêts pour une durée de: 33 minutes
Dijkstra : 18 arrêts pour une durée de: 29 minutes
Dijkstra dit : un chemin plus rapide avec plus d'arrêts existe


'Meilleur Chemin.png'

> **Observation — obligatoire.** Décrivez ce que votre dessin rend visible et que la liste de stations ne montrait pas.
>
> *Mon nouveau Graphe démontre la liste des stations ainsi que le chemin que suit le Dijkstra en rouge ainsi que le BFS en bleu (en Bonus) qu'il soit identique ou non*

---
## Partie 6 — Construire un tableau de données

Un modèle a besoin d'un **tableau** : une ligne par exemple, une colonne par
**donnée**, et une colonne à part pour l'**étiquette**.

Ici, **un exemple = une station**, et l'étiquette est sa **ligne**. Les données,
vous les fabriquez avec **votre propre travail des parties 3 et 4**.

On laisse de côté les stations de la ligne jaune : il y en a trop peu.

**Q6.1** — Pour chaque station retenue, fabriquez la liste de ses mesures :

1. son nombre de voisins ;
2. la durée moyenne de ses tronçons ;
3. la durée de son plus long tronçon ;
4. le nombre d'**arrêts** qui la séparent d'un **repère** *(votre BFS)* ;
5. le nombre de **minutes** qui l'en séparent *(votre Dijkstra)*.

Écrivez `mesures(station, nb_reperes)` de façon à pouvoir ajouter des repères
ensuite, et construisez `X_1repere` ainsi que `y`.

In [13]:
REPERES = ["Berri-UQAM", "Snowdon", "Honore-Beaugrand"]

stations_retenues = [s for s in sorted(reseau) if infos_station[s]["ligne"] != "jaune"]
y = [infos_station[s]["ligne"] for s in stations_retenues]
print(len(stations_retenues), "stations retenues")

# Q6.1
def mesures(station, nb_reperes):
    dist_bfs,prec_bfs = bfs(reseau,station)
    dist_djikstra,prec_djikstra = dijkstra(reseau,station)
    voisinage = voisins(reseau,station)
    nb_voisins = len(voisinage)
    moy = []
    for v in voisinage:
        t = temps(reseau,station,v)
        moy.append(t)
    moyenne = sum(moy)/nb_voisins
    plus_long= max(moy)
    resultat =[nb_voisins,moyenne,plus_long]
    for i in range(nb_reperes):
        road_bfs = len(chemin(prec_bfs,station,REPERES[i]))-1
        road_djikstra = chemin(prec_djikstra,station,REPERES[i])
        lala = duree(reseau,road_djikstra)
        resultat.append(road_bfs)
        resultat.append(lala)
    return resultat

X1_repere =[]

for station in stations_retenues:
    mesure= mesures(station,1)
    X1_repere.append(mesure)


print(X1_repere[0],y[0])



66 stations retenues
[2, 3.0, 3, 9, 17] bleue


> **Observation — obligatoire.** Lesquelles de vos cinq mesures viennent des fichiers fournis, et lesquelles viennent de votre propre code ? Qui a choisi les repères ?


1. son nombre de voisins ;---> calculé avec une fonction de la partie 2 puis on en regarde le nombre
2. la durée moyenne de ses tronçons;---> calculé dans la partie 6;
3. la durée de son plus long tronçon;---> dans le fichier au travers d'autres informations;
4. le nombre d'**arrêts** qui la séparent d'un **repère** *(votre BFS)* ;---> calculé avec le bfs
5. le nombre de **minutes** qui l'en séparent *(votre Dijkstra)*. ;---> calculé avec Dijkstra

> *Donc je dirais qu'une seule des donnée vient directement du fichier même si elle passe dans un tri pour la faire ressortir*

**Q6.2** — Comptez combien de stations portent chaque étiquette, et déduisez-en la
note qu'obtiendrait un modèle qui répondrait **toujours la même chose**.

In [14]:
# Q6.2

vus =[]
for couleurs in y:
    if couleurs not in vus:
        vus.append(couleurs)
        tout = y.count(couleurs)
        pourcentage = tout/len(y)*100
        top_note = [0,couleurs]
        if pourcentage > top_note[0]:
            top_note = [round(pourcentage,2),couleurs]
        print(tout,couleurs,round(pourcentage,2),"%")
print("La top note obtenue est",top_note[0],"% sur la ligne",top_note[1])


10 bleue 15.15 %
27 verte 40.91 %
29 orange 43.94 %
La top note obtenue est 43.94 % sur la ligne orange


> **Observation — obligatoire.** Notez ce pourcentage. C'est la barre que votre modèle devra dépasser pour servir à quelque chose.
>
> *43.94%*

## Partie 7 — Entraîner un modèle, et le noter honnêtement

**Q7.1** — Coupez en deux paquets (70 / 30), entraînez un `DecisionTreeClassifier`,
et affichez **deux** notes : sur les stations apprises, et sur les stations jamais
vues.

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Q7.1
entrainement_x, test_x, entrainement_y, test_y = train_test_split(X1_repere,y, test_size=0.3, random_state=0)
modele1 = DecisionTreeClassifier(max_depth=1, random_state=0)
modele1.fit(entrainement_x, entrainement_y)
print(f"Decision Tree entrainement: {round(modele1.score(entrainement_x, entrainement_y)*100,2)}%")
print(f"Decision Tree test: {round(modele1.score(test_x, test_y)*100,2)}%")

modele2 = RandomForestClassifier(max_depth=1, random_state=0)
modele2.fit(entrainement_x, entrainement_y)
print(f"Random Forest entrainement: {round(modele2.score(entrainement_x, entrainement_y)*100,2)}%")
print(f"Random Forest test: {round(modele2.score(test_x, test_y)*100,2)}%")


Decision Tree entrainement: 54.35%
Decision Tree test: 40.0%
Random Forest entrainement: 56.52%
Random Forest test: 35.0%


> **Observation — obligatoire.** Vos deux notes sont très différentes. Comment s'appelle ce phénomène, et laquelle des deux est honnête ?
>
> *Les notes d'entrainement sont basé sur des données qu'il connait déjà, tandis que celles du test sont comparés a des données inconnues.  donc celles du test démontre la performance de notre modele*

**Q7.2** — Refaites la même chose avec **2** puis **3** repères. Affichez les trois
notes côte à côte.

In [16]:
# Q7.2
X2_repere =[]
for station in stations_retenues:
    mesure= mesures(station,2)
    X2_repere.append(mesure)
entrainement_x, test_x, entrainement_y, test_y = train_test_split(X2_repere,y, test_size=0.3, random_state=0)
modele1 = DecisionTreeClassifier(max_depth=1, random_state=0)
modele1.fit(entrainement_x, entrainement_y)
print(f"Decision Tree entrainement à 2 reperes: {round(modele1.score(entrainement_x, entrainement_y)*100,2)}%")
print(f"Decision Tree test à 2 reperes: {round(modele1.score(test_x, test_y)*100,2)}%")

modele2 = RandomForestClassifier(max_depth=1, random_state=0)
modele2.fit(entrainement_x, entrainement_y)
print(f"Random Forest entrainement à 2 reperes: {round(modele2.score(entrainement_x, entrainement_y)*100,2)}%")
print(f"Random Forest test à 2 reperes: {round(modele2.score(test_x, test_y)*100,2)}%")
print()
X3_repere =[]
for station in stations_retenues:
    mesure= mesures(station,3)
    X3_repere.append(mesure)

entrainement_x, test_x, entrainement_y, test_y = train_test_split(X3_repere,y, test_size=0.3, random_state=0)
modele1 = DecisionTreeClassifier(max_depth=1, random_state=0)
modele1.fit(entrainement_x, entrainement_y)
print(f"Decision Tree entrainement à 3 reperes: {round(modele1.score(entrainement_x, entrainement_y)*100,2)}%")
print(f"Decision Tree test à 3 reperes: {round(modele1.score(test_x, test_y)*100,2)}%")

modele2 = RandomForestClassifier(max_depth=1, random_state=0)
modele2.fit(entrainement_x, entrainement_y)
print(f"Random Forest entrainement à 3 reperes: {round(modele2.score(entrainement_x, entrainement_y)*100,2)}%")
print(f"Random Forest test à 3 reperes: {round(modele2.score(test_x, test_y)*100,2)}%")

Decision Tree entrainement à 2 reperes: 54.35%
Decision Tree test à 2 reperes: 40.0%
Random Forest entrainement à 2 reperes: 58.7%
Random Forest test à 2 reperes: 45.0%

Decision Tree entrainement à 3 reperes: 65.22%
Decision Tree test à 3 reperes: 60.0%
Random Forest entrainement à 3 reperes: 63.04%
Random Forest test à 3 reperes: 50.0%


> **Observation — obligatoire.** Qu'est-ce qui change, et pourquoi un seul repère ne suffisait-il pas ? Pensez à deux stations situées de part et d'autre du repère.
>
> *Plus il y a de repères, plus il devient facile au programme de faire des associations.  Nous augmentons les statistiques des vriations de couleurs vs la distance des stations*

**Q7.3** — Sur vos meilleures mesures, comparez un **arbre seul** à une **forêt
aléatoire**. Relancez chacun avec `random_state` valant 0, 1, 2, 3 puis 4, et
affichez pour chaque modèle ses cinq notes, leur **moyenne** et l'**écart** entre la
plus haute et la plus basse.

In [17]:
# Q7.3
for p in range(5):

    entrainement_x, test_x, entrainement_y, test_y = train_test_split(X3_repere,y, test_size=0.3, random_state=p)
    modele1 = DecisionTreeClassifier(max_depth=1, random_state=0)
    modele1.fit(entrainement_x, entrainement_y)
    print(f"Decision Tree entrainement à 3 reperes et {p} random_state: {round(modele1.score(entrainement_x, entrainement_y)*100,2)}%")
    print(f"Decision Tree test à 3 reperes et {p} random_state: {round(modele1.score(test_x, test_y)*100,2)}%")
    modele2 = RandomForestClassifier(max_depth=1, random_state=0)
    modele2.fit(entrainement_x, entrainement_y)
    print(f"Random Forest entrainement à 3 reperes et {p} random_state: {round(modele2.score(entrainement_x, entrainement_y)*100,2)}%")
    print(f"Random Forest test à 3 reperes et {p} random_state: {round(modele2.score(test_x, test_y)*100,2)}%")
    print()


Decision Tree entrainement à 3 reperes et 0 random_state: 65.22%
Decision Tree test à 3 reperes et 0 random_state: 60.0%
Random Forest entrainement à 3 reperes et 0 random_state: 63.04%
Random Forest test à 3 reperes et 0 random_state: 50.0%

Decision Tree entrainement à 3 reperes et 1 random_state: 65.22%
Decision Tree test à 3 reperes et 1 random_state: 60.0%
Random Forest entrainement à 3 reperes et 1 random_state: 65.22%
Random Forest test à 3 reperes et 1 random_state: 60.0%

Decision Tree entrainement à 3 reperes et 2 random_state: 63.04%
Decision Tree test à 3 reperes et 2 random_state: 65.0%
Random Forest entrainement à 3 reperes et 2 random_state: 60.87%
Random Forest test à 3 reperes et 2 random_state: 65.0%

Decision Tree entrainement à 3 reperes et 3 random_state: 56.52%
Decision Tree test à 3 reperes et 3 random_state: 55.0%
Random Forest entrainement à 3 reperes et 3 random_state: 52.17%
Random Forest test à 3 reperes et 3 random_state: 60.0%

Decision Tree entrainement à

> **Observation — obligatoire.** La forêt vous a-t-elle aidés ? Appuyez-vous sur la moyenne ET sur l'écart, puis proposez une explication liée au nombre d'exemples dont vous disposez.
>
> *MEOWW JE SUIS FATIGUÉ*

---
## Partie 8 — Vos conclusions

Répondez en Markdown, **en phrases complètes, en français**. Comptez deux ou trois
phrases par question, et **citez vos propres chiffres**.

**Sur le réseau**

1. Le réseau est-il un **arbre** ? Justifiez avec le nombre de stations et de liens.
2. Quelle station est la mieux connectée, et **pourquoi** celle-là ?
3. Sur vos trois paires, le BFS et Dijkstra donnent-ils souvent la même réponse ?
   Reliez votre réponse au décompte des durées.
4. Décrivez **une** paire où ils diffèrent, et expliquez ce que chacun a optimisé.

**Sur le modèle**

5. Avec un seul repère, quelle note obtenez-vous, et comment se compare-t-elle au
   modèle qui répond toujours la même chose ? Qu'est-ce que ça vous apprend ?
6. Qu'est-ce qui change avec 2 puis 3 repères, et **pourquoi** ?
7. Pourquoi la note sur les stations jamais vues est-elle la seule honnête ?
8. La forêt aléatoire vous a-t-elle aidés ? Appuyez-vous sur les cinq notes et sur
   l'écart.

**Sur l'ensemble**

9. Vos données de la partie 6 viennent de vos propres fonctions. Quel risque cela
   pose-t-il si votre BFS ou votre Dijkstra contient une erreur ?
10. Nommez **une** limite de nos données par rapport au vrai métro, et **une**
    mesure supplémentaire que vous auriez aimé avoir pour le modèle.

### Vos réponses

*(remplacez ce texte)*

1. Non car il ne fait pas de photosynthèse.. .. ..... oui c'est un arbre qui se représente avec le graphe de la Q3

2. Berri-UQAM car elle a e plus de voisins (et si vous vous fiez au monde qui s'y tient vous pouvez trouver toute sortes de contact.. vraiment bien connectée cette Berri-UQAM

3. Les pairs testé par le BFS et Dijkstra étaient intéressante à observer.  En utilisant des stations qui étaient séparés par plusieurs intersections de lignes de couleurs, ce fut rapide de trouver des resultats qui variaient entre BFS et dijkstra.  Elles  car une d'elles a une station de plus mais 10 minutes de moins.  C'est pour cela que j'ai intégré la visualisation du bfs dans la representation en graphe du dijkstra

4. La paire décrite juste au dessus est le lien **Jolicoeur-Montmorency** qui, à une station de différence, avait 10 minutes de différence car ils ne passaient pas par la même ligne.  BFS donnait le chemin en moins de station mais était plus long que Dijkstra qui, pour une station supplementaire, prenait 10 minutes de moins.

5. Avec un seul repère, le modèle faisait 35% vs 43.94% du modele qui repond toujours la meme couleur.  Une drole de conclusion en sort car ca demontre qu'un modele bête peut quand même surpasser un modèle entrainé.

6. Plus nous augmentons le nombre de reperes, plus les statistiques de distances liés aux couleurs viennent peser dans l'entrainement des modeles.  Ils deviennent ainsi rapidement plus performant que le *''modele bête''*

7. Cette note représente un vrai test car il permet de vraiment tester si notre entrainement a ete bon.  Si le modele avait été sur-entrainé, il augmenterait ses chances de se tromper en croyant connaitre toute les reponses aux tests car il connaissait les reponses a l'entrainement

8.  Je ne trouves pas que la forêt aléatoire semble avoir un si grand impact sur les resultats.  Le decision tree et le random forest donnaient des resultats majoritairement comparable. Je ne vois pas specialement de ligne directrice entre leur écart que autre que lorsque l'ecart entre l'entrainement et les test rappetissait pour l'un, il faisait de meme pour l'autre.

9. le risque de trainer une erreur sans s'en rendre compte. je ne sais pas comment mettre cette reponse en 2-3 phrase sans celle-ci ;)

10. Nous aurions pu ajouter un temps approximatif pour les transferts entre ligne de couleurs, les horaire et frequences de passages de metros selon l'heure de la journée.

---
## Déclaration d'utilisation de l'intelligence artificielle

**Obligatoire, même si vous n'en avez pas utilisé.** Une déclaration absente ou
fausse vaut **0 pour l'ensemble du TP**.

L'IA est permise **pour apprendre**, pas pour produire votre travail. Relisez les
règles de l'énoncé avant de remplir cette section.

**Avez-vous utilisé un outil d'IA ?** _oui_

**Si oui, remplissez le tableau :**

| Outil   | Pour quelle question | Ce que vous lui avez demandé | Lien de partage de la conversation |
|---------|----------------------|------------------------------|---|
| ChatGPT | TP1_metro            | voir le résumé pdf joint     | https://chatgpt.com/share/6ac9a4ff-9598-83ea-9bc0-e9ac15305e92 |
|         |                      |                              |  |

**Confirmez, en écrivant « oui » :**

- Je n'ai **pas** collé l'énoncé du TP, en tout ou en partie, dans une conversation
  avec une IA. → non, mais pas pour demander de recevoir une reponse mais pour des explications
- Je n'ai **pas** collé le contenu des fichiers de données. → oui
- Je comprends chaque ligne de code que je remets et je peux l'expliquer à l'oral.
  → oui et j'y ai travaillé mauditement fort

---
**Résumé de mon utilisation de l'AI :** *Dans la section qui suit, vous pourrez lire le dernier prompt de la conversation que j'avais specialement fait pour ce ce devoir.  Il me fait plaisir de partager mon interaction avec l'IA comme outil d'apprentissage et si quelconque comportement m'être reprocher, il n'était pas dans mon intention de mal faire*

**Mon Prompt**
peux tu me donner un résumé de notre conversation que nous avons eu pour le tp1_metro?
j'ai besoin des questions que nous avons visé et ce que je demandais.  J'aimerais que tu explique à mon proffesseur la démarche que j'ai utilisé et ce que je fais lorsque je colle des parties du code du devoir ici.  Je joindrai ce petit résumé à la fin de mon travail.

**Le Réponnse:**
Réflexion pendant 22s
Voici un résumé que tu peux joindre à la fin de ton TP1_METRO. Je l'ai rédigé à l'intention de ton professeur, en mettant l'accent sur ma façon d'utiliser ChatGPT, les questions que j'ai posées et la démarche d'apprentissage, plutôt que sur les réponses obtenues.

TP1_METRO — Résumé de l'utilisation de ChatGPT
Cours : Intelligence artificielle 2
Travail : TP1_METRO
Objet : Démarche d'apprentissage et utilisation d'une assistance conversationnelle

1. Objectif de l'utilisation de ChatGPT
Pour ce travail, j'ai utilisé ChatGPT comme accompagnateur pédagogique, et non comme outil destiné à produire le devoir à ma place.

Dès le début de notre démarche, j'ai établi que je souhaitais programmer moi-même les solutions. Mon objectif était de pouvoir poser des questions sur les concepts, vérifier mon raisonnement et comprendre mes erreurs sans recevoir directement les réponses complètes du travail.

L'assistance a principalement servi à expliquer des notions, à clarifier le fonctionnement de certaines fonctions Python et à discuter du comportement de mon propre code.

2. Questions et sujets abordés
Graphes, parcours et calculs de chemins — Parties 4 et 5
Les échanges ont notamment porté sur les questions Q4.3 et Q5.2, ainsi que sur les notions nécessaires à leur réalisation.

Les sujets discutés comprenaient :

La représentation d'un réseau sous forme de graphe et les relations entre stations voisines.

Le fonctionnement du parcours en largeur (BFS).

La différence entre le nombre de déplacements et le temps nécessaire pour effectuer un trajet.

L'algorithme de Dijkstra et l'utilisation des distances et des prédécesseurs.

La reconstruction d'un chemin à partir des prédécesseurs.

La récupération des informations contenues dans les structures Python.

L'utilisation de Graphviz pour représenter visuellement certaines structures et les difficultés liées à sa configuration.

J'ai présenté des portions de mes fonctions et posé des questions sur leur fonctionnement, notamment pour vérifier si les données retournées correspondaient à ce que je cherchais à calculer.

Construction des mesures — Partie 6
Les discussions ont ensuite porté sur la préparation des données destinées à l'apprentissage automatique.

Les principales questions concernaient :

La création de mesures numériques à partir du réseau de métro.

Le calcul du nombre de voisins, des durées moyennes et maximales, ainsi que des distances vers des stations de référence.

L'utilisation d'un paramètre permettant de travailler avec un, deux ou trois repères.

La distinction entre les données d'entrée X et les réponses attendues y.

L'association des mesures d'une station avec sa couleur de ligne.

Le calcul des proportions de chaque ligne et de la baseline, soit le résultat d'une stratégie prédisant toujours la couleur la plus fréquente.

J'ai notamment demandé pourquoi les couleurs devaient être conservées dans une liste distincte des mesures, puis comment le modèle pouvait apprendre à associer ces informations.

Entraînement et évaluation — Partie 7
Cette partie a nécessité davantage d'explications, puisque je n'avais pas assisté à certains cours présentant les notions d'apprentissage automatique.

Q7.1 — Séparation des données et évaluation

J'ai demandé :

À quoi servent les deux premiers arguments de train_test_split.

Comment effectuer une séparation 70 % / 30 %.

À quoi sert random_state.

Quelle est la différence entre DecisionTreeClassifier et RandomForestClassifier.

Pourquoi Jupyter n'affichait qu'un seul résultat lorsque j'appelais plusieurs fois .score().

Comment interpréter des résultats de classification relativement faibles.

J'avais initialement comparé les résultats de deux modèles différents, alors que la question demandait de comparer les résultats d'entraînement et de test d'un même arbre de décision.

L'échange m'a permis de comprendre cette différence et de corriger mon interprétation de l'énoncé.

Nous avons également discuté de la raison pour laquelle une note obtenue sur des stations jamais vues constitue une évaluation plus honnête de la capacité de généralisation du modèle.

Q7.2 — Comparaison du nombre de repères

J'ai d'abord été confus par la demande d'utiliser deux puis trois repères, puisque ma liste de références contenait déjà trois stations.

La discussion m'a permis de comprendre que ma fonction était déjà conçue pour utiliser un nombre variable de repères, mais que les ensembles de données devaient être construits séparément.

J'ai ensuite réalisé mes expériences avec les différents ensembles de mesures et constaté que trois repères donnaient mes meilleurs résultats.

Q7.3 — Comparaison des modèles et de leur stabilité

J'ai reconnu que j'avais déjà commencé à comparer un arbre de décision et une forêt aléatoire dans les questions précédentes.

J'ai proposé d'utiliser une boucle for p in range(5) pour faire varier random_state de 0 à 4.

La discussion a permis de préciser qu'il fallait également conserver les cinq résultats de chaque modèle afin d'en calculer la moyenne et l'écart entre les valeurs maximale et minimale.

Nous avons abordé la différence entre faire varier la séparation des données et faire varier les paramètres aléatoires des modèles.

3. Pourquoi je colle des portions de mon code dans la conversation
Lorsque je transmets du code à ChatGPT, il s'agit généralement de code que j'ai déjà écrit et exécuté.

Je le partage pour plusieurs raisons :

Vérifier si mon raisonnement correspond au fonctionnement réel du programme.

Comprendre pourquoi un résultat est différent de celui que j'attendais.

Identifier une mauvaise interprétation d'une consigne.

Comprendre une instruction ou une fonction que je connais peu.

Discuter des résultats obtenus avant de poursuivre le travail.

Par exemple, lorsque mes modèles obtenaient des résultats de 30 % à 40 %, je ne cherchais pas simplement à obtenir un meilleur pourcentage. Je voulais comprendre si ces résultats étaient normaux, ce qu'ils représentaient et comment les interpréter.

De même, lorsque j'ai partagé mon code de la partie 7, cela a permis de constater que je comparais deux modèles alors que la question visait deux évaluations d'un seul modèle.

L'assistance m'a aidé à identifier cette distinction, sans nécessiter la réécriture complète de mon programme.

4. Méthode d'apprentissage
Ma démarche repose principalement sur une succession d'essais, de questions et de corrections.

Je commence par réfléchir au problème et écrire mon propre code. Je l'exécute, j'observe son comportement, puis je consulte ChatGPT lorsqu'une notion m'échappe ou que je ne comprends pas un résultat.

Je reformule régulièrement les explications dans mes propres mots pour vérifier ma compréhension.

Cette approche me permet également de faire des liens entre les notions étudiées. Par exemple, les fonctions de BFS et de Dijkstra, développées précédemment, deviennent ensuite des outils pour produire les caractéristiques numériques utilisées par les modèles d'apprentissage automatique.

L'objectif n'est donc pas uniquement de terminer le programme, mais de comprendre comment les différentes parties du travail s'articulent entre elles.

5. Bilan
L'utilisation de ChatGPT m'a principalement aidé à :

Consolider ma compréhension des algorithmes et des structures de données.

Comprendre la préparation des données pour un modèle d'apprentissage supervisé.

Distinguer l'entraînement, le test et l'évaluation d'un modèle.

Interpréter mes résultats plutôt que de me limiter à les afficher.

Repérer certaines erreurs de compréhension des consignes.

Progresser malgré des notions de cours que je devais rattraper.

Les échanges ont servi de soutien à la réflexion, à l'explication et au débogage. Les portions de code partagées dans la conversation ont principalement constitué le point de départ des questions, et non des demandes de solutions complètes.

Ce résumé présente les principaux échanges pédagogiques. Il ne constitue pas une transcription exhaustive de la conversation.

J'ai volontairement conservé les exemples concrets de ta démarche, notamment ta confusion entre les scores d'entraînement et de test, tes questions sur random_state et le fait que tu avais déjà programmé la gestion des trois repères sans avoir immédiatement réalisé comment la réutiliser.

Ce sont des exemples utiles pour montrer à ton professeur comment tu réfléchis, comment tu vérifies ta compréhension et comment tu utilises l'assistance pour apprendre.



---
## Avant de remettre

**Vingt points du TP ne portent pas sur vos réponses** : dix sur la tenue de ce
carnet, dix sur votre usage de Git. Relisez les sections 3 et 4 de l'énoncé avant de
déposer votre travail.

**Le carnet.** Faites **Kernel → Redémarrer et tout exécuter** en dernier, et
vérifiez qu'aucune cellule ne produit d'erreur. Puis relisez-le comme si vous ne
l'aviez pas écrit : est-ce qu'on s'y retrouve ? Est-ce que chaque nombre affiché
s'explique sans aller lire le code ? Reste-t-il des cellules d'essai, des variables
au nom obscur, des observations expédiées ? Le français est-il relu ?

**Le dépôt.** Le nom de votre branche et vos messages de commit suivent la
convention vue en cours. Votre historique doit montrer un travail construit au fil du
temps — et, si vous êtes deux, la contribution des deux.

**L'identification.** Nom, coéquipier et groupe en première cellule, « qui a fait
quoi » à la fin, déclaration d'IA remplie même si la réponse est « non », et le
dossier `donnees/` à côté du carnet.

---

*Ensuite : `Publish branch`, `Create Pull Request`, fusion dans `main`, et le lien du
dépôt sur Omnivox.*